# 07_lightgbm.ipynb

Converted from your cell-by-cell rebuild guide.

## Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import joblib, json
from pathlib import Path
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import (
    classification_report, f1_score,
    precision_recall_curve, auc, roc_auc_score
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

COINS = ["Bitcoin","Ethereum","Dogecoin"]
lgb_models = {}

def load_feature_cols(coin: str):
    path = Path(f"feature_cols_{coin.lower()}.csv")
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run Notebook 3 first.")
    return pd.read_csv(path)["feature"].tolist()

def time_split(df, val_frac=0.15, test_frac=0.15):
    n = len(df)
    return (
        df.iloc[:int(n*(1-val_frac-test_frac))].copy(),
        df.iloc[int(n*(1-val_frac-test_frac)):int(n*(1-test_frac))].copy(),
        df.iloc[int(n*(1-test_frac)):].copy(),
    )


## Cell 2 — Train LightGBM per coin

In [2]:
for coin in COINS:
    feat_cols = load_feature_cols(coin)
    df = pd.read_csv(f"features_{coin.lower()}.csv", parse_dates=["date"]).sort_values("date")
    train, val, test = time_split(df)

    X_train = train[feat_cols].values;  y_train = train["target"].values.astype(int)
    X_val   = val[feat_cols].values;    y_val   = val["target"].values.astype(int)
    X_test  = test[feat_cols].values;   y_test  = test["target"].values.astype(int)

    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    print(f"{coin}: neg={neg}, pos={pos}, ratio={neg/pos:.2f} | features={len(feat_cols)}")

    param_dist = {
        "n_estimators": [100, 200, 400, 600],
        "max_depth": [3, 4, 5, 6, -1],
        "learning_rate": [0.01, 0.05, 0.1],
        "num_leaves": [15, 31, 63, 127],
        "subsample": [0.6, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "reg_alpha": [0, 0.1, 0.5],
        "reg_lambda": [0, 1.0, 5.0],
        "min_child_samples": [10, 20, 50],
    }
    tscv = TimeSeriesSplit(n_splits=4)
    lgb_base = lgb.LGBMClassifier(
        objective="binary",
        is_unbalance=True,
        random_state=42,
        verbose=-1,
    )
    rs = RandomizedSearchCV(
        lgb_base, param_dist, n_iter=30, cv=tscv,
        scoring="f1", random_state=42, refit=True, n_jobs=-1, verbose=0
    )
    rs.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.log_evaluation(period=-1)]
    )

    best_lgb = rs.best_estimator_
    print(f"{coin} best params: {rs.best_params_}")

    val_prob = best_lgb.predict_proba(X_val)[:, 1]
    best_thr = max(
        np.linspace(0.3, 0.7, 41),
        key=lambda t: f1_score(y_val, (val_prob >= t).astype(int))
    )

    test_prob = best_lgb.predict_proba(X_test)[:, 1]
    y_pred = (test_prob >= best_thr).astype(int)
    prec, rec, _ = precision_recall_curve(y_test, test_prob)
    pr_auc = auc(rec, prec)
    roc = roc_auc_score(y_test, test_prob)
    f1 = f1_score(y_test, y_pred)

    print(f"\n{'='*50}")
    print(f"LightGBM | {coin}")
    print(classification_report(y_test, y_pred, target_names=["Down", "Up"]))
    print(f"ROC-AUC: {roc:.4f}   PR-AUC: {pr_auc:.4f}   F1(Up): {f1:.4f}")

    lgb_models[coin] = {
        "model": best_lgb,
        "threshold": best_thr,
        "feature_cols": feat_cols,
        "metrics": {"f1_up": f1, "roc_auc": roc, "pr_auc": pr_auc}
    }

print("\n✅ LightGBM complete for all coins.")


Bitcoin: neg=1489, pos=1336, ratio=1.11 | features=60
Bitcoin best params: {'subsample': 1.0, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 31, 'n_estimators': 100, 'min_child_samples': 50, 'max_depth': 4, 'learning_rate': 0.01, 'colsample_bytree': 1.0}

LightGBM | Bitcoin
              precision    recall  f1-score   support

        Down       0.00      0.00      0.00       277
          Up       0.54      1.00      0.70       329

    accuracy                           0.54       606
   macro avg       0.27      0.50      0.35       606
weighted avg       0.29      0.54      0.38       606

ROC-AUC: 0.5089   PR-AUC: 0.5404   F1(Up): 0.7037
Ethereum: neg=707, pos=675, ratio=1.05 | features=51
Ethereum best params: {'subsample': 0.8, 'reg_lambda': 0, 'reg_alpha': 0.5, 'num_leaves': 63, 'n_estimators': 200, 'min_child_samples': 10, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.8}

LightGBM | Ethereum
              precision    recall  f1-score   support

        Dow

## Cell 3 — Feature importance (LightGBM gain)

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, coin in zip(axes, COINS):
    feat_cols = lgb_models[coin]["feature_cols"]
    importances = pd.Series(
        lgb_models[coin]["model"].feature_importances_,
        index=feat_cols
    ).nlargest(15).sort_values()
    importances.plot.barh(ax=ax, color="#a855f7", edgecolor="white")
    ax.set_title(f"{coin} — LightGBM Feature Gain (Top 15)")
plt.suptitle("LightGBM Feature Importances", fontsize=14)
plt.tight_layout()
plt.savefig("lgb_feature_importances.png", dpi=150)
plt.show()


## Cell 4 — Save LightGBM models

In [4]:
for coin in COINS:
    joblib.dump(lgb_models[coin]["model"], f"lgb_model_{coin.lower()}.joblib")
    print(f"Saved lgb_model_{coin.lower()}.joblib  threshold={lgb_models[coin]['threshold']:.2f}")

Saved lgb_model_bitcoin.joblib  threshold=0.30
Saved lgb_model_ethereum.joblib  threshold=0.30
Saved lgb_model_dogecoin.joblib  threshold=0.30
